In [2]:
import glob
import os
from moviepy.editor import ImageSequenceClip
import re

In [4]:
work_folder = r'E:\05-Finale runs\zug_100m\figures\map_0m'
prefix_name = 'map'
video_name = f'{prefix_name}.mp4'
fps = 10

# Get all images
images = glob.glob(os.path.join(glob.escape(work_folder), "*.png"))

# Define a function to extract the numerical part of the filename
def extract_number(filename):
    # Use regular expression to find all numbers in the filename
    basename = os.path.basename(filename)
    trimmed = basename.replace(prefix_name, "")
    numbers = re.findall(r'\d+', trimmed)
    # Convert the first found number to an integer
    return int(numbers[0]) if numbers else 0

# Sort the list of images based on the extracted number
sorted_images = sorted(images) #, key=extract_number

# Create a clip from the images
clip = ImageSequenceClip(sorted_images, fps=fps)

# Crop to even dimensions (1270x672)
w, h = clip.size
clip = clip.crop(x1=0, y1=0, x2=w - (w % 2), y2=h - (h % 2))

clip.write_videofile(
     os.path.join(work_folder, video_name),
     fps=fps,
     codec="libx264",
     audio=False,
     ffmpeg_params=[
        "-pix_fmt", "yuv420p",
        "-profile:v", "baseline",
        "-level", "3.1",
        "-movflags", "+faststart"
     ]
)
print("Video created successfully.")

t:  66%|██████▋   | 562/848 [00:41<00:04, 64.30it/s, now=None]

Moviepy - Building video E:\05-Finale runs\zug_100m\figures\map_0m\map.mp4.
Moviepy - Writing video E:\05-Finale runs\zug_100m\figures\map_0m\map.mp4



t:  66%|██████▋   | 562/848 [00:55<00:04, 64.30it/s, now=None]

Moviepy - Done !
Moviepy - video ready E:\05-Finale runs\zug_100m\figures\map_0m\map.mp4
Video created successfully.


In [4]:
sorted(images)

[]

In [25]:
import os
import glob
import re
from PIL import Image
from moviepy.editor import ImageSequenceClip

work_folder1 = r'E:\05-Finale runs\geneva_100m\figures\transect_[34500,11500]_[41500,22500]'
work_folder2 = r'E:\05-Finale runs\geneva_100m\figures\transect_[42000,9000]_[50000,20000]'
prefix_name = 'transect'
video_name = f'{prefix_name}_stacked.mp4'
fps = 4

# Get all images for both sets
images1 = glob.glob(os.path.join(glob.escape(work_folder1), "*.png"))
images2 = glob.glob(os.path.join(glob.escape(work_folder2), "*.png"))

# Function to extract numbers from filenames for sorting
def extract_number(filename):
    basename = os.path.basename(filename)
    trimmed = basename.replace(prefix_name, "")
    numbers = re.findall(r'\d+', trimmed)
    return int(numbers[0]) if numbers else 0

# Sort images
images1 = sorted(images1)
images2 = sorted(images2)

# Ensure both sets have the same number of images
if len(images1) != len(images2):
    images2 = images2[:len(images1)]
    #raise ValueError("Both sets must have the same number of images.")

# Create stacked images
stacked_images = []
for img1_path, img2_path in zip(images1, images2):
    img1 = Image.open(img1_path)
    img2 = Image.open(img2_path)

    # Make sure widths match
    if img1.width != img2.width:
        # Resize the second image to match the width of the first
        img2 = img2.resize((img1.width, int(img2.height * img1.width / img2.width)))

    # Create a new image with height = sum of both heights
    stacked_img = Image.new('RGB', (img1.width, img1.height + img2.height))
    stacked_img.paste(img1, (0, 0))
    stacked_img.paste(img2, (0, img1.height))

    stacked_images.append(stacked_img)

# Save stacked images temporarily (optional, or convert directly to clip)
temp_folder = os.path.join(work_folder1, 'temp_stacked')
os.makedirs(temp_folder, exist_ok=True)
stacked_paths = []
for i, img in enumerate(stacked_images):
    path = os.path.join(temp_folder, f'stacked_{i:03d}.png')
    img.save(path)
    stacked_paths.append(path)

# Create video
clip = ImageSequenceClip(stacked_paths, fps=fps)
clip.write_videofile(os.path.join(work_folder1, video_name), fps=fps, codec="libx264", audio=False)
print("Stacked video created successfully.")


Moviepy - Building video E:\05-Finale runs\geneva_100m\figures\transect_[34500,11500]_[41500,22500]\transect_stacked.mp4.
Moviepy - Writing video E:\05-Finale runs\geneva_100m\figures\transect_[34500,11500]_[41500,22500]\transect_stacked.mp4



Moviepy - Done !
Moviepy - video ready E:\05-Finale runs\geneva_100m\figures\transect_[34500,11500]_[41500,22500]\transect_stacked.mp4
Stacked video created successfully.
